### Experiment + config

In [ ]:
import sys
sys.path.append("../..")
sys.path.append("..")
import os, yaml, torch
from pathlib import Path
thisfiledir = os.getcwd()
expdir = thisfiledir
savedir = expdir + '/model_training/onn_online_training'
# Model imports
import exp_model
# Select GPU as device if possible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
# For multi workers when dataloading
torch.multiprocessing.set_start_method('spawn')

# Set environment variables for display
os.environ["DISPLAY"] = ":0"
os.environ["XAUTHORITY"] = os.path.expanduser("~/.Xauthority")
os.environ["XDG_SESSION_TYPE"] = "x11"
# optional sanity check
print("DISPLAY =", os.environ.get("DISPLAY"))
print("XAUTHORITY =", os.environ.get("XAUTHORITY"))

layer, [train_wid_all, train_wid_one] = exp_model.exp(exp= 11000, widget = True)

zoom, Npix = layer.zoom, layer.Npix 
img_size, slm_size = layer.img_size, layer.slm_size
cam_pad, pat_pad = layer.cam_pad, layer.pat_pad
batch_layout, batch_stacks = layer.batch_layout, layer.batch_stacks
shape_labeled = layer.shape_labeled
Nmux = batch_layout[0]*batch_layout[1]
Npix_x, Npix_y = Npix[0], Npix[1]
Npixtot = Npix_x * Npix_y
Nin = shape_labeled[0]*shape_labeled[1]
os.chdir(expdir)

### Training surrogate model

In [ ]:
import sys
sys.path.append('.')
from pathlib import Path
from config_loader import load_training_config
from surrogate_model_training import train_optical
from models import Surrogate_OpticalNet_Unet
from utils import build_mnist_loaders

cfg, _ = load_training_config("config.yaml")
sur_cfg = cfg["pre_training_surrogate_model"]
train_loader, val_loader = build_mnist_loaders(batch_size=cfg["data"]["batch_size"], augment=cfg["data"]["augment"])
surrogate = Surrogate_OpticalNet_Unet(enc_canvas_hw=tuple(sur_cfg["HW_input_limit"]))
optical_layer = layer

# (Optional) live-reload convenience during iterations
%load_ext autoreload
%autoreload 2

# Matplotlib inline 
%matplotlib inline

history = train_optical(
    model=surrogate,
    layer=optical_layer,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda",
    epochs=sur_cfg["epochs"],
    max_iters=sur_cfg["max_iters"],
    eval_every=sur_cfg["eval_every"],
    lr=sur_cfg["lr"],
    accum_steps=sur_cfg.get("accum_steps", 8),
    weight_decay=sur_cfg["weight_decay"],
    plot_live=sur_cfg["plot_live"],
    run_name=sur_cfg["run_name"],
    ckpt_dir=sur_cfg["ckpt_dir"],
    save_best=sur_cfg["save_best"],
    resume_from=sur_cfg["resume_from"],
    phase_mask_dim=tuple(sur_cfg["phase_mask_dim"]),
    phase_mask_sigma=sur_cfg["phase_mask_sigma"],
    HW_input_limit=tuple(sur_cfg["HW_input_limit"]),
    visualize_samples=sur_cfg["visualize_samples"],
    visualize_every=sur_cfg["visualize_every"],
    ckpt_every=sur_cfg["ckpt_every"],
    quiet_mode=sur_cfg["quiet_mode"],
)

### ONN training with Physics-aware algorithm

In [ ]:
import sys
sys.path.append('.')

# (Optional) live-reload convenience during iterations
%load_ext autoreload
%autoreload 2 

# Matplotlib inline 
%matplotlib inline
from optical_onn_training import train
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Path to the updated config.yaml (kept in the onn_online_training folder)
config_path = 'config.yaml'

# You can override parameters by passing a dict (or leave None to use config.yaml as-is)
overrides = None
train(layer=layer, config_path=config_path, overrides=overrides)

### ONN training with Physics-aware algorithm (Code-Class Readout)

In [ ]:
import os, sys
ROOT = "."
sys.path.insert(0, ROOT)
os.environ["PYTHONPATH"] = ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")

# (Optional) live-reload convenience during iterations
%load_ext autoreload
%autoreload 2

# Matplotlib inline
%matplotlib inline

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from optical_onn_training_code_class_readout import train

config_path = "config_code_class_readout.yaml"
overrides = None
train(layer=layer, config_path=config_path, overrides=overrides)


### ONN training with Physics-aware algorithm (MIX)

In [ ]:
import os, sys
ROOT = "."
sys.path.insert(0, ROOT)
os.environ["PYTHONPATH"] = ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")

# (Optional) live-reload convenience during iterations
%load_ext autoreload
%autoreload 2

# Matplotlib inline
%matplotlib inline

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from optical_onn_training_mix import train

config_path = "config_mix.yaml"
overrides = None
train(layer=layer, config_path=config_path, overrides=overrides)


### ONN training with Physics-aware algorithm (MIX)_Face

In [ ]:
import os, sys
ROOT = "."
sys.path.insert(0, ROOT)
os.environ["PYTHONPATH"] = ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")

# (Optional) live-reload convenience during iterations
%load_ext autoreload
%autoreload 2

# Matplotlib inline
%matplotlib inline

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from optical_onn_training_mix_face_linear import train

config_path = "config_mix_face.yaml"
overrides = None
train(layer=layer, config_path=config_path, overrides=overrides)


### ONN training with Physics-aware algorithm (MIX) VLM

In [ ]:
import os, sys
ROOT = "."
sys.path.insert(0, ROOT)
os.environ["PYTHONPATH"] = ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")

# (Optional) live-reload convenience during iterations
%load_ext autoreload
%autoreload 2

# Matplotlib inline
%matplotlib inline

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from optical_onn_training_mix_VLM import train

config_path = "config_mix_VLM.yaml"
overrides = None
train(layer=layer, config_path=config_path, overrides=overrides)
